# Predicción de resultados de partidos (XGBoost)

Pipeline canónico del TP Integrador que reproduce la comparativa de la Sección 5.2 del informe (`informe/TPIG2.md`) usando el código reutilizable de `src/mineria`.

- Recetas: `ta` (Team Attributes), `pa` (Player Attributes), `pa_ta`, `odds` (cuotas B365) y `combined` (PA + TA + cuotas B365 + rachas 5/10/15).
- Umbral de rentabilidad (breakeven): `1 / 1.85 ≈ 54.05%`.

In [1]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / ".git").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / "src"))

import pandas as pd
from mineria.config import BREAKEVEN, MODELS_DIR
from mineria.pipeline import run_experiment


In [2]:
results = {}
for recipe in ["ta", "pa", "pa_ta", "odds", "combined"]:
    results[recipe] = run_experiment(recipe)


## Comparativa de modelos

Los cuatro modelos de la Sección 5.2 del informe (Team Attributes, Player Attributes, cuotas y Combinado) más la combinación intermedia PA+TA.

In [3]:
comparativa = pd.DataFrame(
    [
        {"Conjunto": "Team Attributes", "recipe": "ta"},
        {"Conjunto": "Player Attributes", "recipe": "pa"},
        {"Conjunto": "Player + Team", "recipe": "pa_ta"},
        {"Conjunto": "Cuotas (B365)", "recipe": "odds"},
        {"Conjunto": "Combinado (PA+TA+cuotas+racha)", "recipe": "combined"},
    ]
)
comparativa["Accuracy train"] = comparativa["recipe"].map(
    lambda r: results[r]["train_accuracy"]
)
comparativa["Accuracy test"] = comparativa["recipe"].map(
    lambda r: results[r]["accuracy"]
)
comparativa["Breakeven 54.05%"] = comparativa["recipe"].map(
    lambda r: "supera" if results[r]["accuracy"] > results[r]["breakeven"] else "no alcanza"
)
comparativa.drop(columns=["recipe"])


,Conjunto,Accuracy train,Accuracy test,Breakeven 54.05%
0,Team Attributes,0.5295,0.5054,no alcanza
1,Player Attributes,0.6086,0.5200,no alcanza
2,Player + Team,0.6087,0.5192,no alcanza
3,Cuotas (B365),0.5259,0.5219,no alcanza
4,Combinado (PA+TA+cuotas+racha),0.5977,0.5242,no alcanza


## Persistencia de modelos

Se guardan los modelos `pa_ta` y `combined` junto con sus columnas de features y el imputador (medianas ajustadas sobre entrenamiento), para reutilizarlos en inferencia.

In [4]:
import joblib

for recipe, out in [("pa_ta", "modelo_pred_pa_ta.joblib"), ("combined", "modelo_pred_pa_ta_streaks.joblib")]:
    r = results[recipe]
    joblib.dump(
        {
            "recipe": r["recipe"],
            "model": r["model"],
            "feature_columns": r["feature_columns"],
            "imputer": r["imputer"],
        },
        MODELS_DIR / out,
    )
print("Modelos guardados en", MODELS_DIR)


Modelos guardados en D:\dev\Mineria\models


## Análisis del modelo Combinado

Importancia de variables y comportamiento frente al umbral de rentabilidad.

In [5]:
importances = pd.Series(
    results["combined"]["model"].feature_importances_,
    index=results["combined"]["feature_columns"],
)
importances.sort_values(ascending=False).head(20)


B365A                                 0.017640
B365H                                 0.016474
B365D                                 0.004537
home_player_5_overall_rating          0.002866
home_team_api_id_defenceAggression    0.002576
home_player_1_overall_rating          0.002554
home_player_4_overall_rating          0.002512
away_player_4_overall_rating          0.002454
home_player_10_overall_rating         0.002413
home_streak_15                        0.002328
away_player_3_overall_rating          0.002250
home_player_2_overall_rating          0.002234
away_player_8_potential               0.002230
home_player_4_interceptions           0.002183
home_player_10_positioning            0.002149
home_player_1_gk_kicking              0.002102
away_player_6_overall_rating          0.002094
home_player_1_potential               0.002047
away_player_4_potential               0.002030
home_player_7_gk_diving               0.002005
dtype: float32

In [6]:
for name in ["ta", "pa", "pa_ta", "odds", "combined"]:
    acc = results[name]["accuracy"]
    marca = "supera" if acc > results[name]["breakeven"] else "no alcanza"
    print(f"{name:8s} acc_test={acc:.4f}  {marca} el breakeven ({BREAKEVEN:.4f})")


ta       acc_test=0.5054  no alcanza el breakeven (0.5405)
pa       acc_test=0.5200  no alcanza el breakeven (0.5405)
pa_ta    acc_test=0.5192  no alcanza el breakeven (0.5405)
odds     acc_test=0.5219  no alcanza el breakeven (0.5405)
combined acc_test=0.5242  no alcanza el breakeven (0.5405)
